# 05. 서브에이전트와 태스크 위임

## 학습 목표
- 서브에이전트가 해결하는 문제(컨텍스트 블로트)를 이해한다
- `SubAgent` dict와 `CompiledSubAgent`로 서브에이전트를 정의한다
- 빌트인 general-purpose 서브에이전트를 이해하고 오버라이드한다
- 컨텍스트 전파와 네임스페이스 키를 활용한다
- 멀티 서브에이전트 파이프라인 패턴을 구현한다

In [1]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

환경 설정 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [3]:
# OpenAI gpt-4.1 모델 설정
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

print(f"모델 설정 완료: {model.model_name}")

/Users/jhj/Desktop/2026_1_sds_ax_advanced/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델 설정 완료: gpt-5.4-mini


---
## 1. 서브에이전트가 필요한 이유

### 컨텍스트 블로트(Context Bloat) 문제

에이전트가 도구를 사용할 때마다 **입력/출력이 컨텍스트 윈도우에 쌓입니다**:
- 웹 검색 결과 (수천 토큰)
- 파일 내용 읽기 (수백~수천 줄)
- 데이터베이스 쿼리 결과

이 중간 결과들이 메인 에이전트의 컨텍스트를 가득 채우면, 정작 중요한 정보를 놓칠 수 있습니다.

### 서브에이전트의 해결 방식

![서브에이전트 컨텍스트 격리](../assets/images/subagent_context.png)

메인 에이전트는 **500 토큰짜리 요약만** 받으므로 컨텍스트가 깔끔하게 유지됩니다.

### 서브에이전트 사용 기준

| 상황 | 서브에이전트 사용 |
|------|------------------|
| 다단계 작업으로 중간 결과가 많음 | ✅ 사용 |
| 전문 지식/도구가 필요한 도메인 | ✅ 사용 |
| 다른 모델이 더 적합한 작업 | ✅ 사용 |
| 단순하고 한 번에 끝나는 작업 | ❌ 불필요 |
| 중간 결과가 메인 에이전트에 필요 | ❌ 불필요 |

---
## 2. SubAgent 정의하기 (dict 기반)

`SubAgent`는 딕셔너리 형태로 정의합니다.

### 필수 필드
| 필드 | 타입 | 설명 |
|------|------|------|
| `name` | `str` | 고유 식별자 |
| `description` | `str` | 역할 설명 (메인 에이전트가 호출 판단에 사용) |
| `system_prompt` | `str` | 서브에이전트 지침 |
| `tools` | `list` | 사용할 도구 목록 |

### 선택 필드
| 필드 | 타입 | 설명 |
|------|------|------|
| `model` | `str` | 모델 오버라이드 (`"provider:model"`) |
| `middleware` | `list` | 추가 미들웨어 |
| `interrupt_on` | `dict` | Human-in-the-Loop 설정 |
| `skills` | `list[str]` | 스킬 소스 경로 |

In [4]:
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
) -> dict:
    """인터넷에서 정보를 검색합니다.

    Args:
        query: 검색할 질문 또는 키워드
        max_results: 반환할 최대 결과 수
        topic: 검색 주제 카테고리
        include_raw_content: 원본 콘텐츠 포함 여부
    """
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


# 리서치 서브에이전트 정의
research_subagent = {
    "name": "researcher",
    "description": "인터넷에서 주제를 심층 조사하고 핵심 정보를 요약합니다. 리서치가 필요한 질문에 사용하세요.",
    "system_prompt": """당신은 전문 리서처입니다.
인터넷 검색을 통해 정확한 정보를 수집하고, 핵심만 추출하여 간결하게 요약합니다.
결과는 항상 한국어로 작성하며, 출처를 함께 표기합니다.
최종 결과는 500단어 이내로 요약하세요.""",
    "tools": [internet_search],
}

print(f"서브에이전트 정의 완료: {research_subagent['name']}")
print(f"설명: {research_subagent['description'][:50]}...")

서브에이전트 정의 완료: researcher
설명: 인터넷에서 주제를 심층 조사하고 핵심 정보를 요약합니다. 리서치가 필요한 질문에 사용하세요...


In [5]:
# 서브에이전트를 포함하는 메인 에이전트 생성
main_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 프로젝트 매니저입니다.
사용자의 요청을 분석하고, 필요하면 전문 서브에이전트에게 작업을 위임합니다.
서브에이전트의 결과를 종합하여 최종 답변을 작성합니다.
한국어로 응답하세요.""",
    subagents=[research_subagent],
)

print("메인 에이전트 생성 완료 (서브에이전트: researcher)")

메인 에이전트 생성 완료 (서브에이전트: researcher)


In [6]:
# 서브에이전트를 활용하는 질문
result = main_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "2026년 AI Agent Harness 에 대해 기술 트렌드를 조사 후 보고서로 작성해주세요.",
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

# 2026년 AI Agent Harness 기술 트렌드 보고서

## 1. 요약
2026년의 AI Agent Harness는 단순한 “에이전트 실행 코드”가 아니라, **에이전트를 안전하게 만들고, 연결하고, 평가하고, 운영하는 통합 플랫폼**으로 진화하고 있습니다.  
핵심은 모델 자체보다 **오케스트레이션, 관측성, 평가, 보안, 메모리, 도구 연동**입니다. 특히 기업 환경에서는 “얼마나 똑똑한가”보다 **얼마나 통제 가능하고 재현 가능하며 감사 가능한가**가 도입 여부를 좌우합니다.

---

## 2. AI Agent Harness의 의미
여기서 Agent Harness는 보통 아래를 포괄합니다.

- 에이전트 실행 및 상태 관리
- 도구 호출(tool use)과 외부 시스템 연동
- 워크플로 / 그래프 기반 오케스트레이션
- 평가(Eval)와 회귀 테스트
- 트레이싱, 로깅, 모니터링
- 메모리 및 컨텍스트 관리
- 안전장치, 정책 제어, 권한 관리
- 멀티에이전트 조정

즉, 에이전트의 “두뇌”보다 **실제로 제품으로 만들기 위한 운영 인프라**에 가깝습니다.

---

## 3. 2026년 핵심 기술 트렌드

### 1) Harness의 표준화 및 플랫폼화
에이전트를 개별 코드로 만드는 시대에서, **통합 하니스 플랫폼**으로 묶는 방향이 강해졌습니다.  
이제 경쟁 포인트는 모델 호출이 아니라, 실행 제어, 상태 추적, 비용 관리, 실패 복구까지 포함한 운영 능력입니다.

**의미**
- 에이전트 개발이 프레임워크 중심에서 플랫폼 중심으로 이동
- 기업은 자체 하니스 또는 상용 플랫폼을 통해 통제력 확보
- 팀 간 재사용 가능한 공통 실행 계층이 중요해짐

---

### 2) 오케스트레이션의 고도화
단순한 순차 tool-calling을 넘어서, **상태 기반 워크플로, 그래프 구조, 조건 분기, 멀티에이전트 조정**이 일반화되고 있습니다.

**핵심 변화**
- 단일 에이전트 중심 → 역할 분담형 에이전트 구조
- 정적 프롬프트 → 동적 실

---
## 3. CompiledSubAgent — 커스텀 LangGraph 그래프 연결

미리 컴파일된 LangGraph 그래프를 서브에이전트로 사용할 수 있습니다.  
복잡한 워크플로(조건 분기, 반복 등)가 필요한 경우에 유용합니다.

In [7]:
from deepagents import CompiledSubAgent

# 별도의 에이전트를 CompiledSubAgent로 래핑하는 예시
# 실제로는 create_deep_agent()로 만든 그래프도 사용 가능
custom_graph = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="당신은 데이터 분석 전문가입니다. 데이터를 수집하고 통계적으로 분석하여 인사이트를 도출합니다.",
)

# CompiledSubAgent로 래핑
data_analyst_subagent: CompiledSubAgent = {
    "name": "data-analyst",
    "description": "데이터를 수집하고 분석하여 통계적 인사이트를 제공합니다.",
    "runnable": custom_graph,
}

print(f"CompiledSubAgent 정의 완료: {data_analyst_subagent['name']}")

CompiledSubAgent 정의 완료: data-analyst


---
## 4. General-Purpose 서브에이전트

Deep Agents는 별도로 정의하지 않아도 **빌트인 general-purpose 서브에이전트**를 자동 제공합니다.

### 기본 동작
- 메인 에이전트와 **같은 시스템 프롬프트** 사용
- 메인 에이전트와 **같은 도구** 접근 가능
- 메인 에이전트와 **같은 모델** 사용 (이 노트북에서는 OpenAI `gpt-4.1`)
- 메인 에이전트의 **스킬** 상속

### 오버라이드
`name="general-purpose"`로 서브에이전트를 정의하면 기본 동작을 덮어씁니다.

In [ ]:
# general-purpose 서브에이전트 오버라이드 예시
custom_gp_agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="당신은 멀티태스크 코디네이터입니다.",
    subagents=[
        research_subagent,
        {
            # 이름을 "general-purpose"로 설정하면 빌트인을 오버라이드
            "name": "general-purpose",
            "description": "범용 에이전트로, 리서치 외의 일반적인 멀티스텝 작업을 처리합니다.",
            "system_prompt": "당신은 범용 어시스턴트입니다. 주어진 작업을 단계별로 처리하세요.",
            "tools": [internet_search],
        },
    ],
)

print("general-purpose 서브에이전트를 오버라이드한 에이전트 생성 완료")

---
## 5. 컨텍스트 전파

런타임 컨텍스트는 자동으로 모든 서브에이전트에 전파됩니다.  
`context_schema`로 구조를 정의하고, `config`의 `context` 키로 값을 전달합니다.

In [8]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

# 컨텍스트 스키마를 가진 에이전트
context_agent = create_deep_agent(
    model=model,
    system_prompt="사용자 맞춤 어시스턴트입니다. 한국어로 응답하세요.",
    subagents=[research_subagent],
    context_schema={"user_id": str, "language": str},
    checkpointer=MemorySaver(),
)

# 컨텍스트와 함께 실행
result = context_agent.invoke(
    {"messages": [HumanMessage(content="내 최근 관심사에 맞는 뉴스를 찾아주세요.")]},
    config={
        **{
            "configurable": {"thread_id": "ctx-test"},
            "context": {
                "user_id": "user-123",
                "language": "ko",
            },
        },
        **lf_config,
    },
)

print(result["messages"][-1].content)

최근 관심사를 알려주는 단서가 없어 바로 맞춤 뉴스를 찾기 어렵습니다.

아래 중 하나만 보내주세요:
- 최근 관심사 키워드 3~5개
- 관심 분야: 예) IT, 경제, AI, 주식, 부동산, 게임, 연예, 스포츠
- 원하는 뉴스 스타일: 요약 위주 / 깊게 분석 / 최신 속보

예시:
- “AI, 반도체, 미국 경제”
- “주식과 경제 뉴스”
- “요즘 제 관심사는 여행, 테크, 축구예요”

주시면 그에 맞춰 바로 골라드리겠습니다.


### 네임스페이스 키로 서브에이전트별 컨텍스트 전달

`"서브에이전트이름:키"` 형식을 사용하면, 특정 서브에이전트에만 전달되는 설정을 추가할 수 있습니다.

```python
config = {
    "context": {
        "user_id": "user-123",             # 모든 에이전트에 전파
        "researcher:max_depth": 3,          # researcher에만 전달
        "data-analyst:strict_mode": True,   # data-analyst에만 전달
    }
}
```

---
## 6. 멀티 서브에이전트 파이프라인

여러 서브에이전트를 조합하여 **수집 → 분석 → 작성** 파이프라인을 구성할 수 있습니다.

![멀티 서브에이전트 파이프라인](../assets/images/subagent_pipeline.png)

In [9]:
# 멀티 서브에이전트 파이프라인
pipeline_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 프로젝트 코디네이터입니다.
사용자의 요청을 분석하고, 적절한 서브에이전트에게 순서대로 작업을 위임합니다:
1. 먼저 data-collector로 정보를 수집합니다.
2. 수집된 정보를 data-analyzer에게 전달하여 분석합니다.
3. 분석 결과를 report-writer에게 전달하여 보고서를 작성합니다.
최종 보고서를 사용자에게 전달합니다. 한국어로 응답하세요.""",
    subagents=[
        {
            "name": "data-collector",
            "description": "다양한 소스에서 원시 데이터와 정보를 수집합니다.",
            "system_prompt": """당신은 데이터 수집 전문가입니다.
주어진 주제에 대해 인터넷 검색을 수행하고, 관련 데이터를 최대한 수집합니다.
수집한 데이터를 구조화하여 반환하세요.""",
            "tools": [internet_search],
        },
        {
            "name": "data-analyzer",
            "description": "수집된 데이터를 분석하여 핵심 인사이트를 추출합니다.",
            "system_prompt": """당신은 데이터 분석 전문가입니다.
제공된 데이터에서 패턴, 트렌드, 핵심 인사이트를 추출합니다.
분석 결과를 불릿 포인트로 정리하세요.""",
            "tools": [],
        },
        {
            "name": "report-writer",
            "description": "분석 결과를 바탕으로 전문적인 보고서를 작성합니다.",
            "system_prompt": """당신은 테크니컬 라이터입니다.
분석 결과를 바탕으로 명확하고 읽기 쉬운 보고서를 작성합니다.
보고서는 다음 구조를 따릅니다: 개요 → 핵심 발견 → 결론""",
            "tools": [],
        },
    ],
)

print("멀티 서브에이전트 파이프라인 에이전트 생성 완료")

멀티 서브에이전트 파이프라인 에이전트 생성 완료


In [10]:
# 파이프라인 실행
result = pipeline_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "2026년 생성형 AI 시장 동향에 대한 간단한 보고서를 작성해 주세요.",
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

# 2026년 생성형 AI 시장 동향

## 요약
2026년 생성형 AI 시장은 **실험 단계에서 대규모 업무 적용 단계로 전환**되고 있습니다.  
단순 챗봇을 넘어 검색, 문서작성, 고객응대, 코딩, 데이터 분석 등 핵심 업무 전반으로 확산되며, 시장의 관심도는 모델 성능보다 **보안, 비용, 운영 안정성, 규제 대응**으로 이동하고 있습니다.

## 핵심 동향
- **주요 활용 분야**
  - 고객응대/컨택센터: 상담 요약, 자동 응답, 상담원 보조
  - 문서 업무: 보고서 초안, 회의록, 계약서 검토
  - 개발 생산성: 코드 생성, 테스트, 디버깅
  - 사내 지식검색: 매뉴얼, 정책, 내부 문서 질의응답
  - 마케팅/콘텐츠: 광고 문안, 캠페인 소재, 개인화 콘텐츠

- **기술 트렌드**
  - 멀티모달 확산: 텍스트뿐 아니라 이미지, 음성, 영상까지 처리
  - AI 에이전트화: 단순 생성에서 업무 실행형 AI로 진화
  - RAG 및 업무 특화 모델 확대: 사내 데이터 연계 강화
  - 온디바이스/프라이빗 AI 수요 증가: 보안과 비용 문제 대응
  - 경량화·최적화 중요성 확대: 응답 속도와 추론 비용이 핵심 경쟁 요소

- **규제/거버넌스**
  - 저작권, 개인정보, 설명 가능성, 책임소재 이슈가 확대
  - 금융, 의료, 공공 분야는 감사 가능성, 기록 관리, 인간 승인 체계가 중요
  - 기업은 모델뿐 아니라 데이터 관리와 검증 절차를 함께 구축해야 함

- **기업 도입 과제**
  - 환각과 품질 편차
  - 보안 및 데이터 유출 우려
  - 기존 시스템 연동 난이도
  - ROI 입증 부담
  - 현업·IT·보안·법무 간 협업 필요

## 한국 시장 시사점
- 제조, 금융, 유통, 공공 서비스에서 활용 여지가 큼
- 문서 중심 업무와 상담 업무에서 빠른 성과 기대
- 글로벌 모델 활용과 함께 한국어·국내 규제에 맞는 최적화가 중요
- 경쟁력은 데이터 품질, 보안 체계, 현업 적용력에서 갈릴 가능성이 높음

## 결론
2026

---
## 7. 베스트 프랙티스

### 1. 명확한 description 작성
메인 에이전트는 `description`을 보고 어떤 서브에이전트를 호출할지 결정합니다.  
-> 서브에이전트의 역할과 사용 시기를 명확하게 기술하세요.

### 2. 최소 도구 원칙
서브에이전트에는 **필요한 도구만** 제공하세요.  
불필요한 도구는 컨텍스트를 낭비하고 오동작의 원인이 됩니다.

### 3. 간결한 결과 반환
서브에이전트의 시스템 프롬프트에 **"결과를 간결하게 요약하라"**는 지침을 포함하세요.  
그래야 메인 에이전트가 효율적으로 결과를 처리할 수 있습니다.

### 4. 적절한 모델 선택
작업 복잡도에 따라 서브에이전트마다 다른 모델을 사용할 수 있습니다.  
이 노트북에서는 OpenAI `gpt-4.1` 모델을 사용합니다.  
다양한 프로바이더의 모델을 유연하게 선택할 수 있습니다:
- 단순 수집 -> 가벼운 모델 (예: `gpt-4.1-mini`)
- 깊은 분석 -> 강력한 모델 (예: `gpt-4.1`, `anthropic:claude-sonnet-4`)

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| SubAgent | dict 기반 정의: `name`, `description`, `system_prompt`, `tools` |
| CompiledSubAgent | 커스텀 LangGraph 그래프를 `runnable`로 연결 |
| General-Purpose | 빌트인 기본 서브에이전트 (메인과 동일한 설정) |
| 컨텍스트 전파 | `context_schema` + `config["context"]` |
| 네임스페이스 키 | `"에이전트이름:키"` 형식으로 서브에이전트별 설정 |
| 파이프라인 패턴 | collector → analyzer → writer |

## 다음 단계
→ **[06_memory_and_skills.ipynb](./06_memory_and_skills.ipynb)**: 장기 메모리와 스킬을 활용하여 에이전트의 지식을 확장합니다.